# Phase O — OBTAIN
## Download FRED macro data + Yahoo Finance market prices

**OSEMN Step 1**: Pull raw data from external sources and cache locally.

Data sources:
- **FRED** (Federal Reserve): yield curve, credit spreads, GDP, CPI, unemployment, PMI proxies, M2
- **Yahoo Finance (yfinance)**: SPY, TLT, GLD, LQD, HYG, BIL, NVDA, WDC

All series IDs, tickers, and date ranges come from `config/config.yaml`.

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src.obtain.fred_loader import FREDLoader
from src.obtain.market_loader import MarketLoader

cfg = load_config('../config/config.yaml')
print(f"Project: {cfg['project']['name']}")
print(f"Date range: {cfg['data']['fred']['start_date']} → present")

## 1.1 FRED Series Catalogue
Check what we're downloading before hitting the API.

In [ ]:
fred_loader = FREDLoader(cfg)
fred_loader.describe()

## 1.2 Download FRED Data
First run downloads and caches. Subsequent runs load from `data/raw/fred/`.

In [ ]:
fred_data = fred_loader.load(force_refresh=False)  # set True to re-download

print(f"\nLoaded {len(fred_data)} series:")
for name, s in fred_data.items():
    print(f"  {name:35s} {s.index[0].date()} → {s.index[-1].date()}  "
          f"({len(s)} obs, {s.isna().mean():.1%} NaN)")

## 1.3 Quick Visual Check — Key Macro Series

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
key_series = ['yield_curve_10y2y', 'credit_spread_baa', 'unemployment',
               'cpi', 'gdp_real', 'industrial_production']

for ax, name in zip(axes.flatten(), key_series):
    if name in fred_data:
        fred_data[name].plot(ax=ax, linewidth=1)
        ax.set_title(name, fontsize=10)
        ax.axhline(0, color='red', lw=0.5, ls='--')

plt.suptitle('Raw FRED Series (original frequency)', fontsize=13)
plt.tight_layout()
plt.show()

## 1.4 Market Instruments Catalogue

In [ ]:
market_loader = MarketLoader(cfg)
market_loader.describe()

## 1.5 Download Market Data

In [ ]:
market_data = market_loader.load(force_refresh=False)

print(f"Loaded {len(market_data)} instruments:")
for ticker, df in market_data.items():
    print(f"  {ticker:6s} {df.index[0].date()} → {df.index[-1].date()}  "
          f"({len(df)} daily obs)")

## 1.6 ETF Price History

In [ ]:
etfs = list(cfg['data']['market']['etfs'].keys())
fig, ax = plt.subplots(figsize=(16, 6))

for ticker in etfs:
    if ticker in market_data:
        prices = market_data[ticker]['Close']
        norm = prices / prices.iloc[0]  # normalise to 1
        norm.plot(ax=ax, label=ticker, linewidth=1.2)

ax.set_yscale('log')
ax.set_title('ETF Normalised Price History (log scale)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## Summary
- FRED data downloaded and cached to `data/raw/fred/`
- Market data downloaded and cached to `data/raw/market/`
- **Next**: `02_scrub.ipynb` — clean, align, normalise, engineer features